In [5]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# --- Dataframe ---
nhanes = pd.read_csv("./data/nhanes-data.csv", delimiter="\t")

# --- Visualisierung ---
# pd.plotting.scatter_matrix(nhanes)
# plt.show()

# fig, axs = plt.subplots(2, 4, figsize=(10, 5))
# plt.tight_layout(pad=3.5)
# for column, ax in zip(nhanes.columns, axs.flatten()):
#     nhanes.boxplot(by="Gender", column=column, ax=ax)
# plt.show()

# fig, axs = plt.subplots(2, 4, figsize=(10, 5))
# plt.tight_layout(pad=3.5)
# for column, ax in zip(nhanes.columns, axs.flatten()):
#     ax.set_title(column)
#     ax.hist(nhanes.query("Gender == 1")[column], alpha=.5, label="Mann", bins=50)
#     ax.hist(nhanes.query("Gender == 0")[column], alpha=.5, label="nicht Mann", bins=50)
# fig.suptitle("Histogram grouped by Gender")
# plt.legend()
# plt.show()

column = "Height"
thetas = np.linspace(nhanes[column].min(), nhanes["Height"].max(), 100)

# --- Receiver Operating Characteristics -> ROC ---
def ROC(thetas, column="Height"):
    # --- Positve Values -> P ---
    P = nhanes.query("Gender == 1").shape[0]

    # --- Negative Values -> N ---
    N = nhanes.query("Gender == 0").shape[0]

    def _ROC(theta):
        # --- True Positives -> TP ---
        TP = nhanes.query(f"Gender == 1 and {column} >= {theta}").shape[0]
        
        # --- False Positives -> FP ---
        FP = nhanes.query(f"Gender == 0 and {column} >= {theta}").shape[0]

        # --- True Positve Rate ---
        TPR = TP / P

        # --- False Positve Rate ---
        FPR = FP / N

        return FPR, TPR
    
    return np.array([_ROC(theta) for theta in thetas])

# --- Visualisierung ---
# plt.plot(*ROC(thetas).T, marker="o")
# plt.plot((0, 1), (0, 1), '--', c="black")

# plt.title("ROC Kurve")
# plt.xlabel("FPR")
# plt.ylabel("TPR")
# plt.show()

# --- reverse ROC Curve -> Jouden Index (Js) ---
X, Js = np.flipud(ROC(thetas)).T

# --- Under the ROC Curve -> AUC ---
AUC = np.trapz(y=Js, x=X, dx=1/Js.shape[0])